***Imports***

In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import operator
from pathlib import Path
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler


 ***Carregar o dataset***

In [6]:
csv_base = Path.cwd().parent.parent.parent / "data" 
csv_path_limpo = Path.cwd().parent.parent.parent / "data" / "dataset_completo_limpo.v1.csv"
df = pd.read_csv(csv_path_limpo)

In [ ]:
df.head()

,Year,Quarter,Month,DayofMonth,DayOfWeek,Reporting_Airline,OriginAirportID,OriginCityMarketID,OriginStateFips,DestAirportID,DestCityMarketID,DestStateFips,CRSElapsedTime,Distance,DistanceGroup,CRSDepHour,CRSDepMinute,CRSArrHour,CRSArrMinute,Target
0,2022,1,1,14,5,YX,11066,31066,39,11278,30852,51,88,323,2,12,24,13,52,1
1,2022,1,1,15,6,YX,11066,31066,39,11278,30852,51,88,323,2,12,24,13,52,1
2,2022,1,1,16,7,YX,11066,31066,39,11278,30852,51,88,323,2,12,24,13,52,1
3,2022,1,1,17,1,YX,11066,31066,39,11278,30852,51,88,323,2,12,24,13,52,1
4,2022,1,1,18,2,YX,11066,31066,39,11278,30852,51,88,323,2,12,24,13,52,1


In [7]:
df['Reporting_Airline'] = df['Reporting_Airline'].astype("category")
df['OriginAirportID'] = df['OriginAirportID'].astype("category")
df['OriginCityMarketID'] = df['OriginCityMarketID'].astype("category")
df['OriginStateFips'] = df['OriginStateFips'].astype("category")
df['DestAirportID'] = df['DestAirportID'].astype("category")
df['DestCityMarketID'] = df['DestCityMarketID'].astype("category")
df['DestStateFips'] = df['DestStateFips'].astype("category")

In [8]:
df.isnull().sum()

Year                  0
Quarter               0
Month                 0
DayofMonth            0
DayOfWeek             0
Reporting_Airline     0
OriginAirportID       0
OriginCityMarketID    0
OriginStateFips       0
DestAirportID         0
DestCityMarketID      0
DestStateFips         0
CRSElapsedTime        0
Distance              0
DistanceGroup         0
CRSDepHour            0
CRSDepMinute          0
CRSArrHour            0
CRSArrMinute          0
Target                0
dtype: int64

**Split dos dados**

In [9]:
y = df['Target']
X = df.drop(columns=['Target'])

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42,stratify=y_train)

In [13]:
len(X_train[X_train['CRSElapsedTime']<=0])

5

In [14]:
mask = X_train["CRSElapsedTime"] > 0

X_train = X_train.loc[mask]
y_train = y_train.loc[mask]

Retirar coluna do ano

In [ ]:
X_train.drop(columns=['Year'],inplace=True)
X_val.drop(columns=['Year'],inplace=True)
X_test.drop(columns=['Year'],inplace=True)

In [17]:
X_train.columns

Index(['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek',
       'Reporting_Airline', 'OriginAirportID', 'OriginCityMarketID',
       'OriginStateFips', 'DestAirportID', 'DestCityMarketID', 'DestStateFips',
       'CRSElapsedTime', 'Distance', 'DistanceGroup', 'CRSDepHour',
       'CRSDepMinute', 'CRSArrHour', 'CRSArrMinute'],
      dtype='object')

In [22]:
ct = ColumnTransformer([
        ('number_normalization', StandardScaler(), ['Quarter', 'Month','DayofMonth','DayOfWeek','CRSElapsedTime','Distance','DistanceGroup',
            'CRSDepHour','CRSDepMinute','CRSArrHour','CRSArrMinute'])
    ], remainder='passthrough')
ct.fit_transform(X_train)
ct.transform(X_val)
ct.transform(X_test)

array([[0.4245761921006296, 0.7088088534477571, 0.25791449307324793, ...,
        11433, 31295, 26],
       [-1.382388334666689, -1.6438833364892511, -0.3127719052309158,
        ..., 12173, 32134, 15],
       [-1.382388334666689, -1.055710289004999, -1.6824192611609088, ...,
        14107, 30466, 4],
       ...,
       [0.4245761921006296, 0.41472232970563105, 1.0568754506990772, ...,
        13930, 30977, 17],
       [-1.382388334666689, -1.349796812747125, 0.3720517727340807, ...,
        11433, 31295, 26],
       [1.328058455484289, 1.2969819009320092, -1.111732862856745, ...,
        14893, 33192, 6]], dtype=object)